In [ ]:
using NCDatasets, Plots, Dates, Downloads

# Data Download

In [ ]:
# 1. Define the specific target date and cycle you found
yyyy = "2026"
mm = "02"
dd = "16"
yyyymmdd = "20260216"
cycle = "18"  # Using the t18z cycle you found
f_hour = "001" # Using n001

println("Attempting to download CBOFS canonical dataset for $yyyymmdd...")

# 2. Construct the direct fileServer URL using the new 2026 format
# dataset=NOAA/CBOFS/MODELS/2026/02/16/cbofs.t18z.20260216.fields.n001.nc
thredds_url = "https://opendap.co-ops.nos.noaa.gov/thredds/fileServer/NOAA/CBOFS/MODELS/$yyyy/$mm/$dd/cbofs.t$(cycle)z.$yyyymmdd.fields.n$f_hour.nc"

local_file = "static_chesapeake_salinity.nc"

try
    print("Downloading from NOAA... ")
    Downloads.download(thredds_url, local_file)
    println("Success!")
    println("File permanently saved to: $local_file")
    println("\nYou can now load this file locally in your JuMP notebook!")
catch e
    println("\nDownload failed. Error: ", e)
end

In [ ]:
local_file = "datafiles/static_chesapeake_salinity.nc"
ds = NCDataset(local_file)

In [ ]:
   
    # 2. Extract Lat/Lon coordinates
    lon = ds["lon_rho"][:]
    lat = ds["lat_rho"][:]
    
    # 3. Extract Surface Salinity
    # Dimensions: [xi_rho, eta_rho, s_rho (depth), ocean_time]
    # We use 'end' for the top depth layer, and '1' for the first/only time step
    salt = ds["salt"][:, :, end, 1] 
    
    # Close the dataset to free memory
    close(ds)
    
    # 4. Filter out land (missing or NaN values)
    # ROMS models use a land mask where values are either 'missing' or massive fill values
    valid_idx = .!ismissing.(salt) .&& .!isnan.(salt) .&& (salt .< 100)
    
    # Ensure lon/lat shape aligns with the salinity mask. NetCDF variables
    # sometimes come back as 1D flattened arrays while `salt` is 2D.
    if size(lon) == size(valid_idx) && size(lat) == size(valid_idx)
        lon_valid = lon[valid_idx]
        lat_valid = lat[valid_idx]
    elseif ndims(lon) == 1 && length(lon) == length(vec(valid_idx))
        lon2 = reshape(lon, size(valid_idx))
        lat2 = reshape(lat, size(valid_idx))
        lon_valid = lon2[valid_idx]
        lat_valid = lat2[valid_idx]
    else
        # Fallback: linearize both and apply linear mask
        lon_valid = vec(lon)[vec(valid_idx)]
        lat_valid = vec(lat)[vec(valid_idx)]
    end
    salt_valid = Float64.(salt[valid_idx])
    
    println("Plotting $(length(salt_valid)) ocean data points...")
    
    # 5. Generate the Plot
    # We use GR backend's dense scatter plot to correctly render the curvilinear grid
    gr()
    p = scatter(lon_valid, lat_valid, zcolor=salt_valid,
             m=(1.8, :square, stroke(0)), # Small borderless squares mimic a heatmap
             c=:viridis,                  # Standard oceanographic color map
             clims=(0, 35),               # Estuary salinity ranges from 0 (fresh) to ~35 (ocean)
             title="Chesapeake Bay Surface Salinity",
             xlabel="Longitude", 
             ylabel="Latitude",
             colorbar_title="Salinity (PSU)",
             legend=false,
             size=(600, 800),             # Taller aspect ratio fits the Bay perfectly
             framestyle=:box,
             grid=true)

In [ ]:
# 1. Define the date range
start_date = Date(2026, 2, 16)
dates = [start_date + Day(i) for i in 0:4]
cycles = ["00", "06", "12", "18"]  # All 4 daily model runs

println("Attempting to download all available CBOFS files for 5 days...")
println("WARNING: This may download up to ~24 GB of data.")

total_ok = 0
total_fail = 0

for date in dates
    yyyy = string(year(date))
    mm = lpad(string(month(date)), 2, "0")
    dd = lpad(string(day(date)), 2, "0")
    yyyymmdd = yyyy * mm * dd

    for cycle in cycles
        # 2. Get the daily THREDDS catalog for this specific date
        catalog_url = "https://opendap.co-ops.nos.noaa.gov/thredds/catalog/NOAA/CBOFS/MODELS/$yyyy/$mm/$dd/catalog.html"

        f_hours = String[]
        try
            # Download catalog HTML into memory
            catalog_tmp = IOBuffer()
            Downloads.download(catalog_url, catalog_tmp)
            html = String(take!(catalog_tmp))

            # Match: cbofs.t18z.20260216.fields.n001.nc
            pat = Regex("cbofs\\.t$(cycle)z\\.$yyyymmdd\\.fields\\.n(\\d{3})\\.nc")
            f_hours = unique([m.captures[1] for m in eachmatch(pat, html)])
            sort!(f_hours)

            if isempty(f_hours)
                println("No files found in catalog for $yyyymmdd cycle t$(cycle)z.")
                continue
            end

            println("\nFound $(length(f_hours)) files for $yyyymmdd (t$(cycle)z): ", join(f_hours, ", "))
        catch e
            println("Could not read catalog for $yyyymmdd. Error: ", e)
            continue
        end

        # 3. Download every available f_hour for that cycle
        for f_hour in f_hours
            thredds_url = "https://opendap.co-ops.nos.noaa.gov/thredds/fileServer/NOAA/CBOFS/MODELS/$yyyy/$mm/$dd/cbofs.t$(cycle)z.$yyyymmdd.fields.n$f_hour.nc"
            local_file = "chesapeake_salinity_$(yyyymmdd)_t$(cycle)z_n$(f_hour).nc"

            # Skip if we already downloaded it (useful if script crashes and you restart)
            if isfile(local_file)
                println("  Skipping $local_file (Already exists)")
                continue
            end

            try
                print("  Downloading $yyyymmdd t$(cycle)z n$f_hour... ")
                Downloads.download(thredds_url, local_file)
                println("Success!")
                global total_ok += 1
            catch e
                println("Failed. Error: ", e)
                global total_fail += 1
            end
        end
    end
end

println("\nDone. Successful new downloads: $total_ok | Failed: $total_fail")

# Animation of long dataset

In [ ]:
# Collect all downloaded CBOFS files and build timestamps from filename
pat = r"^chesapeake_salinity_(\d{8})_t(\d{2})z_n(\d{3})\.nc$"
files = filter(f -> occursin(pat, f), readdir())

if isempty(files)
    error("No files found matching chesapeake_salinity_YYYYMMDD_tCCz_nFFF.nc")
end

frames = NamedTuple[]
for f in files
    m = match(pat, f)
    ymd = m.captures[1]
    cyc = parse(Int, m.captures[2])   # cycle hour
    fh  = parse(Int, m.captures[3])   # forecast/nowcast hour
    dt = DateTime(ymd, dateformat"yyyymmdd") + Hour(cyc) + Hour(fh)
    push!(frames, (file=f, dt=dt))
end
sort!(frames, by = x -> x.dt)

# Helper to load one snapshot
function load_surface_salinity(file)
    ds = NCDataset(file)
    lon = ds["lon_rho"][:]
    lat = ds["lat_rho"][:]
    salt = ds["salt"][:, :, end, 1]
    close(ds)

    valid_idx = .!ismissing.(salt) .&& .!isnan.(salt) .&& (salt .< 100)

    if size(lon) == size(valid_idx) && size(lat) == size(valid_idx)
        lon_valid = lon[valid_idx]
        lat_valid = lat[valid_idx]
    elseif ndims(lon) == 1 && length(lon) == length(vec(valid_idx))
        lon2 = reshape(lon, size(valid_idx))
        lat2 = reshape(lat, size(valid_idx))
        lon_valid = lon2[valid_idx]
        lat_valid = lat2[valid_idx]
    else
        lon_valid = vec(lon)[vec(valid_idx)]
        lat_valid = vec(lat)[vec(valid_idx)]
    end

    return lon_valid, lat_valid, Float64.(salt[valid_idx])
end

gr()
anim = @animate for fr in frames
    lon_valid, lat_valid, salt_valid = load_surface_salinity(fr.file)

    scatter(
        lon_valid, lat_valid,
        marker_z = salt_valid,          # <- use marker_z for scalar coloring
        m = (1.8, :square, stroke(0)),
        c = :viridis,
        clims = (0, 35),
        colorbar = true,                # <- force colorbar on
        colorbar_title = "Salinity (PSU)",
        title = "Chesapeake Bay Surface Salinity\n$(Dates.format(fr.dt, dateformat"yyyy-mm-dd HH:MM")) UTC",
        xlabel = "Longitude",
        ylabel = "Latitude",
        legend = false,
        size = (600, 800),
        framestyle = :box,
        grid = true
    )
end

mp4(anim, "chesapeake_salinity_animation.mp4", fps = 6)

In [ ]:
lat_min, lat_max = 36.75, 37.5

gr()
anim_zoom = @animate for fr in frames
    lon_valid, lat_valid, salt_valid = load_surface_salinity(fr.file)

    roi = (lat_valid .>= lat_min) .& (lat_valid .<= lat_max)

    if any(roi)
        scatter(
            lon_valid[roi], lat_valid[roi],
            marker_z = salt_valid[roi],
            m = (1.8, :square, stroke(0)),
            c = :viridis,
            clims = (0, 35),
            colorbar = true,
            colorbar_title = "Salinity (PSU)",
            title = "Chesapeake Bay Surface Salinity (Zoomed)\n$(Dates.format(fr.dt, dateformat"yyyy-mm-dd HH:MM")) UTC",
            xlabel = "Longitude",
            ylabel = "Latitude",
            ylims = (lat_min, lat_max),
            legend = false,
            size = (700, 500),
            framestyle = :box,
            grid = true
        )
    else
        plot(
            title = "No ocean points in ROI\n$(Dates.format(fr.dt, dateformat"yyyy-mm-dd HH:MM")) UTC",
            xlabel = "Longitude",
            ylabel = "Latitude",
            ylims = (lat_min, lat_max),
            legend = false,
            size = (700, 500),
            framestyle = :box,
            grid = true
        )
    end
end

mp4(anim_zoom, "chesapeake_salinity_animation_zoom.mp4", fps = 6)